# Modern Object Detection: Open-Vocabulary & VLM Grounding

## Learning Objectives
* Understand the transition from traditional object detection to open-vocabulary and text-grounded models.
* Perform zero-shot detection on arbitrary classes using Google's **PaliGemma** Vision-Language Model.
* Fine-tune a VLM using **LoRA (Low-Rank Adaptation)** on a custom research dataset (e.g. brain tumor scans).
* Decode PaliGemma coordinate output tokens and display visual bounding boxes.

In [ ]:
# Install Keras 3, Keras Hub, and Kaggle Hub
!pip install -q --upgrade keras-hub
!pip install -q --upgrade keras
!pip install -q --upgrade kagglehub opencv-python

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
import keras_hub
import jax
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import urllib.request

devices = jax.devices()
print("Available devices:", devices)

is_cpu_only = all(d.platform == "cpu" for d in devices)
if is_cpu_only:
    print("⚠️ WARNING: Running on CPU Fallback mode. Training will be constrained.")
    BATCH_SIZE = 1 # Keep to 1 to run on CPU without crashing
    EPOCHS = 1     # 1 epoch demonstration on CPU
    MODEL_PRESET = "pali_gemma_3b_224"
else:
    print("⚡ Accelerator detected! Running on high-performance backend.")
    BATCH_SIZE = 4
    EPOCHS = 5
    MODEL_PRESET = "pali_gemma_3b_mix_224"


## 1. Zero-shot Open-Vocabulary Object Detection

Traditional detection models (like YOLO or Faster R-CNN) can only detect the fixed set of classes they were trained on (like the 80 classes in COCO).

PaliGemma is a **Vision-Language Model (VLM)**. It allows us to perform zero-shot detection of *arbitrary* objects by changing our text prompt. The format for detection is `detect [target object]\n`.

In [ ]:
print(f"Downloading and loading preset: {MODEL_PRESET}...")
model = keras_hub.models.PaliGemmaCausalLM.from_preset(MODEL_PRESET)

# Download example image
image_url = "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"
urllib.request.urlretrieve(image_url, "cow.png")
img = Image.open("cow.png").resize((224, 224))

# Test prompt
prompt = "detect cow\n"
output = model.generate({"images": np.array(img), "prompts": prompt})
print("Model output text:", output)


### Decoding Coordinate Tokens

PaliGemma returns bounding boxes as specialized tokens representing relative coordinates (normalized from 0 to 1000): `<locYMIN><locXMIN><locYMAX><locXMAX> label`.

We will parse these tokens with a regular expression, rescale them back to pixels, and draw them using OpenCV.

In [ ]:
import re

def parse_paligemma_output(output_text, img_width, img_height):
    # Coords are serialized in <locXXXX> tags
    pattern = r"<loc(\d+)><loc(\d+)><loc(\d+)><loc(\d+)>\s*(.*)"
    matches = re.findall(pattern, output_text)
    
    boxes = []
    labels = []
    for match in matches:
        ymin, xmin, ymax, xmax = [float(val) / 1000.0 for val in match[:4]]
        label = match[4].strip()
        
        # Rescale normalized coordinates to pixels
        box = [
            int(ymin * img_height),
            int(xmin * img_width),
            int(ymax * img_height),
            int(xmax * img_width)
        ]
        boxes.append(box)
        labels.append(label)
    return boxes, labels

def draw_boxes(image_path, boxes, labels):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    for box, label in zip(boxes, labels):
        ymin, xmin, ymax, xmax = box
        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(img, label, (xmin, max(ymin - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

boxes, labels = parse_paligemma_output(output, 224, 224)
draw_boxes("cow.png", boxes, labels)


## 2. Fine-Tuning PaliGemma with LoRA

To adapt PaliGemma to custom research datasets (like detecting features in biological slides or locating specific regions of interest), we can use Parameter-Efficient Fine-Tuning (PEFT) with **LoRA (Low-Rank Adaptation)**.

We will download a brain tumor detection dataset from Kaggle, format its COCO bounding boxes to PaliGemma coordinate tokens, configure LoRA, and train.

In [ ]:
import kagglehub

print("Downloading Brain Tumor image dataset (COCO format) from Kaggle...")
# Download public dataset anonymously
dataset_path = kagglehub.dataset_download("pkdarabi/brain-tumor-image-dataset-semantic-segmentation")
print("Dataset downloaded to:", dataset_path)


In [ ]:
import json
import glob

# Locate annotations file
annotation_files = glob.glob(os.path.join(dataset_path, "**/_annotations.coco.json"), recursive=True)
if annotation_files:
    with open(annotation_files[0]) as f:
        coco_data = json.load(f)
    
    # Map images
    img_dict = {img["id"]: img for img in coco_data["images"]}
    anno_dict = {}
    for anno in coco_data["annotations"]:
        img_id = anno["image_id"]
        bbox = anno["bbox"] # [xmin, ymin, width, height]
        if img_id not in anno_dict:
            anno_dict[img_id] = []
        anno_dict[img_id].append(bbox)
        
    # Compile a small training batch to fit memory limits
    images_train = []
    prompts_train = []
    targets_train = []
    
    count = 0
    for img_id, bboxes in anno_dict.items():
        img_info = img_dict[img_id]
        file_name = img_info["file_name"]
        
        matches = glob.glob(os.path.join(dataset_path, "**", file_name), recursive=True)
        if not matches: continue
        
        pil_img = Image.open(matches[0]).resize((224, 224))
        w, h = img_info["width"], img_info["height"]
        
        target_str = ""
        for bbox in bboxes:
            xmin, ymin, bw, bh = bbox
            xmax = xmin + bw
            ymax = ymin + bh
            
            # Convert to [0-1000] scale
            ymin_n = int((ymin / h) * 1000)
            xmin_n = int((xmin / w) * 1000)
            ymax_n = int((ymax / h) * 1000)
            xmax_n = int((xmax / w) * 1000)
            
            target_str += f"<loc{ymin_n}><loc{xmin_n}><loc{ymax_n}><loc{xmax_n}> tumor "
            
        images_train.append(np.array(pil_img))
        prompts_train.append("detect brain tumor\n")
        targets_train.append(target_str.strip())
        
        count += 1
        if count >= BATCH_SIZE:
            break
            
    images_train = np.array(images_train)
    print(f"Prepared {len(images_train)} training images.")
else:
    print("Could not find COCO annotations JSON. Check dataset structure.")


In [ ]:
# Setup LoRA (Low-Rank Adaptation) on PaliGemma
model.backbone.enable_lora(rank=4)
model.summary()

model.compile(optimizer=keras.optimizers.Adam(2e-5))

print("Training the model (LoRA layers active)...")
model.fit(
    x={
        "images": images_train,
        "prompts": prompts_train
    },
    y=targets_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)
print("Model training finished!")


In [ ]:
# Test the fine-tuned model
test_img = images_train[0]
output_tuned = model.generate({"images": test_img, "prompts": "detect brain tumor\n"})
print("Tuned model response:", output_tuned)

# Visualize results
Image.fromarray(test_img).save("test_scan.png")
boxes, labels = parse_paligemma_output(output_tuned, 224, 224)
draw_boxes("test_scan.png", boxes, labels)
